In [9]:
import torch
from torch import nn, Tensor
import math
import random

### BERT 模型

BERT 模型是理解式的语言模型，只使用到了 Transformer-Encoder

主要任务是提取文本的特征，理解文本信息

BERT 并不是生成式的语言模型（生成式的语言模型是自回归的）

<br>

### BERT 模型的数据处理

BERT 模型的输入是一对句子对，整理为一下形式：

![](./md-img/bert_input.png)

其中\<seq>表示一个句子的结束，\<cls>用于后续的 NSP 任务中

同时需呀对上面的序列中的随机 token 进行概率 mask，用于后续的 MLM 任务

mask 的规则如下：

- 在序列中随机选取 15% 的 token 进行 mask

- 将选出的 token 按照 80% 的概率替换为 \<mask> 特殊词元

- 10% 的概率替换为随机词

- 10% 的概率保持不变

In [ ]:
# 设计较大的 max_len，保证每个批次中不会有文本信息的丢失
def process_input(sentence_ids1, sentence_ids2, cls_id, seq_id, pad_id, mask_id, max_len):
    data_processed = [cls_id] + sentence_ids1 + [seq_id] + sentence_ids2 + [seq_id]

    # 填充
    if len(data_processed) < max_len:
        data_processed = data_processed + [pad_id] * (max_len - len(data_processed))

    # 随机遮挡 token
    num_masked = int(max_len * 0.15)
    masked_positions = random.sample(range(max_len), num_masked)   # 随机选取的遮挡位置
    masked_tokens = []    # 遮挡位置对应的 token id（做 label 使用）
    masked_weights = []   # 遮挡住的位置是否是有效 token（是否要计入损失）
    for i in masked_positions:
        masked_tokens.append(data_processed[i])
        if data_processed[i] == cls_id or \
           data_processed[i] == seq_id or \
           data_processed[i] == pad_id:
            masked_weights.append(0)
        else:
            masked_weights.append(1)

        # 采取对应的 mask 策略
        if random.random() < 0.8:
            data_processed[i] = mask_id
        else:
            if random.random() < 0.5:
                data_processed[i] = random.choice(data_processed)
    
    return data_processed, masked_positions, masked_tokens, masked_weights

<br>

### MLM 任务

BERT 预训练时的输入为了 MLM 任务特意设计了 mask

MLM 任务就是从编码器输出中选取出 mask 对应位置的输出

通过 MLP 获取得到对 mask token 的预测 logits

<br>

### NSP 任务

使用编码器输出中 \<cls> 对应位置的输出

通过 MLP 获取一个二份类的 logits

正类表示输入的句子对是连续的

负类表示输入的句子对不是连续的

在采样的过程中就可以获取对应的 label

<br>

### BERT预训练模型

![](./md-img/BERT.png)

<br>

### Transformer-Encoder

In [ ]:
''' 多头注意力层 '''
class MultiHeadAttention(nn.Module):
    def __init__(self, query_size: int, key_size: int, value_size: int, hidden_size: int, num_heads: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.linear_q = nn.Linear(query_size, hidden_size)
        self.linear_k = nn.Linear(key_size, hidden_size)
        self.linear_v = nn.Linear(value_size, hidden_size)
        self.linear_o = nn.Linear(hidden_size, hidden_size)
        self.softmax = nn.Softmax(dim=-1)

    # query: (batch_size, num_querys, query_size)
    # key: (batch_size, num_pairs, key_size)
    # value: (batch_size, num_pairs, value_size)
    # mask: (batch_size, num_querys, num_pairs)
    def forward(self, query: Tensor, key: Tensor, value: Tensor, mask: Tensor = None) -> Tensor:
        batch_size = query.shape[0]
        num_querys = query.shape[1]
        num_pairs = key.shape[1]

        query = self.linear_q(query)
        key = self.linear_k(key)
        value = self.linear_v(value)

        query = query.reshape(batch_size, num_querys, self.num_heads, self.head_dim).permute(0, 2, 1, 3).reshape(batch_size * self.num_heads, num_querys, self.head_dim)
        key = key.reshape(batch_size, num_pairs, self.num_heads, self.head_dim).permute(0, 2, 3, 1).reshape(batch_size * self.num_heads, self.head_dim, num_pairs)
        value = value.reshape(batch_size, num_pairs, self.num_heads, self.head_dim).permute(0, 2, 1, 3).reshape(batch_size * self.num_heads, num_pairs, self.head_dim)
        score = torch.bmm(query, key) / math.sqrt(self.head_dim)

        if mask is not None:
            mask = mask.unsqueeze(1).expand(batch_size, self.num_heads, num_querys, num_pairs).reshape(batch_size * self.num_heads, num_querys, num_pairs)
            score = score.masked_fill(mask == False, float('-inf'))

        weight = self.softmax(score)
        output = torch.bmm(weight, value)
        output = output.reshape(batch_size, self.num_heads, num_querys, self.head_dim).permute(0, 2, 1, 3).reshape(batch_size, num_querys, self.hidden_size)
        return self.linear_o(output)

In [ ]:
''' 前馈网络层 '''
class PositionWiseFFN(nn.Module):
    def __init__(self, input_size: int, ffn_size: int):
        super().__init__()
        self.linear1 = nn.Linear(input_size, ffn_size)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(ffn_size, input_size)

    # x: (batch_size, seq_len, model_size)
    def forward(self, x: Tensor) -> Tensor:
        return self.linear2(self.relu(self.linear1(x)))

In [ ]:
''' 残差连接与归一化层 '''
class ResAndNorm(nn.Module):
    def __init__(self, norm_size: int, dropout: float):
        super().__init__()
        self.norm = nn.LayerNorm(norm_size)
        self.dropout = nn.Dropout(dropout)

    # x: 原始数据
    # y: 输出数据
    def forward(self, x: Tensor, y: Tensor) -> Tensor:
        return self.norm(x + self.dropout(y))

In [ ]:
''' Transformer-Encoder-Block '''
class EncoderBlock(nn.Module):
    def __init__(self, model_size: int, num_heads: int, dropout: float, ffn_size: int):
        super().__init__()
        self.attention = MultiHeadAttention(model_size, model_size, model_size, model_size, num_heads)
        self.res_norm1 = ResAndNorm(model_size, dropout)
        self.ffn = PositionWiseFFN(model_size, ffn_size)
        self.res_norm2 = ResAndNorm(model_size, dropout)

    # x: (batch_size, seq_len, model_size)
    def forward(self, x: Tensor, padding_mask: Tensor = None) -> Tensor:
        output = self.res_norm1(x, self.attention(x, x, x, padding_mask))
        return self.res_norm2(output, self.ffn(output))

<br>

### BERT Model

In [ ]:
''' 可学习的位置编码 '''
class PositionEncoder(nn.Module):
    def __init__(self, seq_len, model_size):
        super().__init__()

        self.pe = nn.Parameter(torch.randn(seq_len, model_size))

    # x: (batch_size, seq_len, model_size)
    def forward(self, x):
        return x + self.pe.unsqueeze(0)

In [ ]:
class BERTModel(nn.Module):
    def __init__(self, vocab_size, model_size, max_len, num_blocks, num_heads, dropout, ffn_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, model_size)
        self.position_encoder = PositionEncoder(max_len, model_size)
        self.encoder = nn.Sequential()
        for i in range(num_blocks):
            self.encoder.add_module(f'encoder_block{i}', EncoderBlock(model_size, num_heads, dropout, ffn_size))
        
        # MLM 任务
        self.mlm = nn.Sequential(
            nn.Linear(model_size, hidden_size),
            nn.ReLU(),
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, vocab_size)
        )

        # NSP 任务
        self.nsp = nn.Sequential(
            nn.Linear(model_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 2)
        )

    # x: (batch_size, max_len)
    # padding_mask: (batch_size, num_querys, num_pairs)
    # masked_positions: (batch_size, num_masked)
    def forward(self, x, padding_mask=None, masked_positions=None):
        encoder_output = self.encoder(self.position_encoder(self.embedding(x)), padding_mask)

        # Masked Language Modeling
        batch_size, _, model_size = encoder_output.shape
        _, num_masked = masked_positions.shape
        batch_idxs = torch.arange(batch_size).unsqueeze(-1).expand(-1, num_masked).reshape(-1)
        masked_positions = masked_positions.reshape(-1)
        mlm_input = encoder_output[batch_idxs, masked_positions]
        mlm_input = mlm_input.reshape(batch_size, num_masked, model_size)
        mlm_output = self.mlm(mlm_input)

        # Next Sentence Prediction
        nsp_input = encoder_output[:, 0, :]
        nsp_output = self.nsp(nsp_input)

        # encode_output: (batch_size, max_len, model_size)
        # mlm_output: (batch_size, num_masked, vocab_size)
        # nsp_output: (batch_size, 2)
        return encoder_output, mlm_output, nsp_output

<br>

### 预训练

训练过程如下：

- 按照批次进行正向传播

- 分别计算 mlm 任务和 nsp 任务的交叉熵损失

- 使用两个损失的和进行反向传播

- 更新参数

此外可能涉及到很多训练细节（混合精度训练、梯度缩放、梯度裁剪、学习率调度、、、）